# Data Ingestion Pipeline

In [1]:
%pip install pypdf langchain-text-splitters
%pip install "langchain-huggingface==1.2.2" "chromadb" "langchain-chroma==1.1.0" "sentence-transformers"
%pip install  langchain langchain-core langchain-community langchain-ollama

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### Step-1: Chunking

In [2]:
from pathlib import Path
import re

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


SECTION_HEADINGS = [
    "What .* is and what it is used for",
    "Before you take",
    "How to take",
    "Possible side effects",
    "Use in pregnancy and breast-feeding",
    "How to store",
]

HEADING_PATTERN = re.compile(
    r"^(?:\d+\.\s*)?(" + "|".join(SECTION_HEADINGS) + r").*$",
    re.IGNORECASE
)


def split_by_section(text):
    sections = []
    current_section = "Overview"
    current_text = []

    for line in text.splitlines():
        line = line.strip()

        if HEADING_PATTERN.match(line):
            if current_text:
                sections.append((current_section, "\n".join(current_text).strip()))

            current_section = line
            current_text = []
        else:
            current_text.append(line)

    if current_text:
        sections.append((current_section, "\n".join(current_text).strip()))

    return sections


def get_medicine_name(text):
    first_lines = text.splitlines()

    for line in first_lines:
        if line.strip():
            return line.strip()

    return "Unknown"


def chunk_documents(docs_path="docs", chunk_size=1000, chunk_overlap=100):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    all_chunks = []

    for pdf_path in Path(docs_path).glob("*.pdf"):
        print(f"Processing: {pdf_path.name}")

        pages = PyPDFLoader(str(pdf_path)).load()
        text = "\n".join(page.page_content for page in pages)

        medicine_name = get_medicine_name(text)

        sections = split_by_section(text)

        for section, content in sections:
            chunks = splitter.split_text(content)

            for i, chunk in enumerate(chunks):
                chunk_text = f"MEDICINE: {medicine_name}\nSECTION: {section}\n\n{chunk}"

                all_chunks.append({
                    "id": f"{pdf_path.name}::{section}::{i}",
                    "source": pdf_path.name,
                    "medicine": medicine_name,
                    "section": section,
                    "text": chunk_text
                })

    return all_chunks

/var/folders/fd/tjdwkh6x2tl4_t9qrrkhy_2h0000gn/T/ipykernel_51882/3401177121.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/opt/miniconda3/envs/experiment/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step-2: Meta data adding to chunks + convert chunks to langchain Documents

In [5]:
from langchain_core.documents import Document
def building_langchain_document_and_add_metadata(chunks):
    documents = []

    for chunk in chunks:
        document = Document(
            page_content=chunk["text"],
            metadata={
                "source": chunk["source"],
                "medicine": chunk["medicine"],
                "section": chunk["section"],
            }
        )

        documents.append(document)

    print(f"Created {len(documents)} LangChain documents.")
    
    return documents

### Step-3: Embedding 

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6301.98it/s]


Embedding model loaded.


### Step-4: Store documents in ChromaDB + Call Chunking and Embedding here 

In [7]:
from langchain_chroma import Chroma


docs_path="docs"
chunk_size=1000
chunk_overlap=100

chunks = chunk_documents(docs_path,chunk_size,chunk_overlap)
documents= building_langchain_document_and_add_metadata(chunks)

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./chroma_db",
    collection_name="medicine_leaflets",
)

print(f"Stored {len(documents)} documents in ChromaDB.")

Processing: maxpro_20mg_esomeprazole_leaflet.pdf
Processing: fenadin_120mg_fexofenadine_leaflet.pdf
Processing: rolip_10mg_rosuvastatin_leaflet.pdf
Processing: doxicap_100mg_doxycycline_leaflet.pdf
Processing: rolac_10mg_ketorolac_leaflet.pdf
Created 35 LangChain documents.
Stored 35 documents in ChromaDB.


# Data Ingestion pipeline Completed

# Retriver Pipeline Started

# Hybrid Retriever (Semantic + BM25 Keyword)

In [33]:
from langchain_community.retrievers import BM25Retriever

def semantic_search(query,k=5):
    
    # Semantic search
    # way-1:
    semantic_results = vectorstore.similarity_search_with_score(query, k=k)
    
    # way-2:
    # query_embedding = embeddings.embed_query(query)
    # results = vectorstore.similarity_search_by_vector(query_embedding,k=5)
    
    # way-3:
    # retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    # results = retriever.invoke(query)    
    
    # way-4:
    # query_embedding = embeddings.embed_query(query)
    # results = vectorstore.similarity_search_by_vector(query_embedding,k=5)
    
    
    return semantic_results


def keyword_search(query, documents,k=5):
    # BM25 search
    bm25_retriever = BM25Retriever.from_documents(documents)
    bm25_retriever.k = k
    bm25_results = bm25_retriever.invoke(query)
    
    return bm25_results
    
    


def hybrid_search(semantic_results,bm25_results,k=5):

    results = []
    seen = set()

    # Add semantic results
    for document, score in semantic_results:
        doc_id = (
            document.metadata["source"]
            + "::"
            + document.metadata["section"]
            + "::"
            + document.page_content
        )

        if doc_id not in seen:
            results.append(document)
            seen.add(doc_id)

    # Add BM25 results
    for document in bm25_results:
        doc_id = (
            document.metadata["source"]
            + "::"
            + document.metadata["section"]
            + "::"
            + document.page_content
        )

        if doc_id not in seen:
            results.append(document)
            seen.add(doc_id)

    return results

# Add Reranker 

In [16]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

reranker_model = HuggingFaceCrossEncoder( 
    model_name="cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8162.24it/s]


In [34]:
# from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# reranker_model = HuggingFaceCrossEncoder( 
#     model_name="cross-encoder/ms-marco-MiniLM-L-6-v2"
# )


def rerank_documents(query, documents,k=5):
    pairs = []
    for document in documents:
        pair = [query, document.page_content]
        pairs.append(pair)
    scores = reranker_model.score(pairs)
    scored_documents = []
    
    for i, (document, score) in enumerate(zip(documents, scores), start=1):
        scored_documents.append([document, score])
        
    scored_documents.sort(key=lambda item: item[1], reverse=True)
    results = []
    
    scored_top_k_document=scored_documents[:k]

    for document, score in scored_top_k_document:
        results.append(document)

    return results






# # With Logs 

# def rerank_documents(query, documents,k=5):
#     pairs = []
#     for document in documents:
#         pair = [query, document.page_content]
#         pairs.append(pair)
#     scores = reranker_model.score(pairs)
#     scored_documents = []
    
#     # print(f"Total Candidate Douments/chunks = {len(documents)}")
#     # print("=" * 50)
#     # for document, score in zip(documents, scores):
#     #     # print("Document:", document.page_content)
#     #     print("Cross-Encoder score:", score)
#     #     print("-" * 50)
#     #     scored_documents.append([document, score])

#     for i, (document, score) in enumerate(zip(documents, scores), start=1):
#         scored_documents.append([document, score])
#         # print(f"Chunk {i}")
#         # print(f"Cross-Encoder Score: {score}")
#         # print("-" * 50)
        
        
#     scored_documents.sort(key=lambda item: item[1], reverse=True)
#     results = []

#     print(f"After giving Score by Reranker model Top {k} Chunks are: ")
#     for document, score in scored_documents[:k]:
#         # print("Document:", document.page_content)
#         # print("Final Score:", score)
#         results.append(document)

#     return results


In [35]:

def create_context(documents):

    context = ""

    for i, document in enumerate(documents):
        # context += f"\nDocument {i + 1}:\n"
        context += document.page_content
        context += "\n\n"

    return context

In [36]:


from langchain_ollama import ChatOllama
llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)



def generate_answer(query, context):


    prompt = f"""
    Answer the question using only the provided context.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    response = llm.invoke(prompt)

    return response.content





# Generation Full Flow

In [37]:
query = " I am 5 years old. can i take doxicap?"



# query_embedding = embeddings.embed_query(query)

Top_K = 5 

semantic_results = semantic_search(query,Top_K)

bm25_results = keyword_search(query, documents)

# Get candidates chunks from Hybrid retriever
candidates = hybrid_search(semantic_results,bm25_results,Top_K)


final_reranked_results = rerank_documents(query,candidates,Top_K)


context = create_context(final_reranked_results)


answer = generate_answer(query,context)

print(f"Ollama final answer=====> {answer}")

Ollama final answer=====> No, you should not take Doxicap if you are a child under 8 years of age.
